In [67]:
import pandas as pd

Retrieved DAM Settlement Point Prices (2023,2024) from ERCOT website with the help of Perplexity Comet App



In [68]:
import pandas as pd
import zipfile
import tempfile
from pathlib import Path


def load_dam_settlement_prices(
    zip_path: str,
    year: int,
    settlement_point_pattern: str = r"^LZ_",
) -> pd.DataFrame:
    """Load ERCOT DAM settlement point prices from a zip file for a given year.

    Parameters
    ----------
    zip_path : str
        Path to the main ERCOT zip file.
    year : int
        Calendar year to filter on (e.g., 2023).
    settlement_point_pattern : str, optional
        Regex pattern for settlement points to filter, by default r"^LZ_" (all load zones).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: datetime, SettlementPoint, SettlementPointPrice, DSTFlag
        containing only rows within the specified calendar year.
    """

    dfs: list[pd.DataFrame] = []

    with zipfile.ZipFile(zip_path, "r") as main_zip:
        nested_zip_files = [f for f in main_zip.namelist() if f.endswith(".zip")]

        for nested_zip_name in sorted(nested_zip_files):
            with tempfile.TemporaryDirectory() as temp_dir:
                nested_zip_bytes = main_zip.read(nested_zip_name)
                nested_zip_path = Path(temp_dir) / nested_zip_name
                nested_zip_path.write_bytes(nested_zip_bytes)

                with zipfile.ZipFile(nested_zip_path, "r") as nested_zip:
                    csv_files = [f for f in nested_zip.namelist() if f.endswith(".csv")]

                    for csv_file in csv_files:
                        with nested_zip.open(csv_file) as file:
                            df = pd.read_csv(file)

                            # keep all SettlementPoint values that begin with "LZ_"
                            df = df[df["SettlementPoint"].astype(str).str.match(settlement_point_pattern)].copy()

                            base_date = pd.to_datetime(df["DeliveryDate"], format="%m/%d/%Y")
                            hours = df["HourEnding"].str.slice(0, 2).astype(int)
                            minutes = df["HourEnding"].str.slice(3, 5).astype(int)
                            df["datetime"] = (
                                base_date
                                + pd.to_timedelta(hours, unit="h")
                                + pd.to_timedelta(minutes, unit="m")
                            )

                            df = df[["datetime", "SettlementPoint", "SettlementPointPrice", "DSTFlag"]]

                            dfs.append(df)

    result = pd.concat(dfs, ignore_index=True)

    start_year = pd.Timestamp(f"{year}-01-01")
    start_next_year = pd.Timestamp(f"{year + 1}-01-01")
    result = result[(result["datetime"] >= start_year) & (result["datetime"] < start_next_year)]

    result = result.sort_values("datetime").reset_index(drop=True)

    return result

In [ ]:
# import pandas as pd
# import zipfile
# from io import BytesIO
# from pathlib import Path


# def load_ancillary_capacity_prices(
#     zip_path: str,
#     year: int,
#     ancillary_type: str = "REGDN",
# ) -> pd.DataFrame:
#     """Load ERCOT DAM clearing prices for capacity from a zip file for a given year.

#     Parameters
#     ----------
#     zip_path : str
#         Path to the main ERCOT zip file.
#     year : int
#         Calendar year to filter on (e.g., 2023).
#     ancillary_type : str, optional
#         Ancillary service type to filter, by default "REGDN".

#     Returns
#     -------
#     pd.DataFrame
#         DataFrame with columns: datetime, ancillary_type, mcpc, dst_flag
#         containing only rows within the specified calendar year.
#     """

#     dfs: list[pd.DataFrame] = []

#     with zipfile.ZipFile(zip_path, "r") as main_zip:
#         nested_zip_files = [f for f in main_zip.namelist() if f.endswith(".zip")]

#         for nested_zip_name in sorted(nested_zip_files):
#             nested_zip_bytes = main_zip.read(nested_zip_name)
            
#             with zipfile.ZipFile(BytesIO(nested_zip_bytes), "r") as nested_zip:
#                 csv_files = [f for f in nested_zip.namelist() if f.endswith(".csv")]

#                 for csv_file in csv_files:
#                     with nested_zip.open(csv_file) as file:
#                         df = pd.read_csv(file)

#                         # Filter by ancillary type
#                         df = df[df["AncillaryType"].astype(str).str.upper() == ancillary_type.upper()].copy()

#                         # Combine DeliveryDate and HourEnding into datetime
#                         base_date = pd.to_datetime(df["DeliveryDate"])
#                         hours = df["HourEnding"].str.slice(0, 2).astype(int)
#                         minutes = df["HourEnding"].str.slice(3, 5).astype(int)
#                         df["datetime"] = (
#                             base_date
#                             + pd.to_timedelta(hours, unit="h")
#                             + pd.to_timedelta(minutes, unit="m")
#                         )

#                         # Select columns
#                         df = df[["datetime", "AncillaryType", "MCPC", "DSTFlag"]]
#                         df.columns = ["datetime", "ancillary_type", "mcpc", "dst_flag"]

#                         dfs.append(df)

#     result = pd.concat(dfs, ignore_index=True)

#     # Filter by year
#     start_year = pd.Timestamp(f"{year}-01-01")
#     start_next_year = pd.Timestamp(f"{year + 1}-01-01")
#     result = result[(result["datetime"] >= start_year) & (result["datetime"] < start_next_year)]

#     result = result.sort_values("datetime").reset_index(drop=True)

#     return result


In [80]:
df_2023 = load_dam_settlement_prices('data/dam_spp_2023.zip', 2023)
# as_2023 = load_ancillary_capacity_prices('data/ancillaryservices_2023.zip', 2023, 'REGDN')
# as_2023 = as_2023.rename(columns={
#     "mcpc": "regdn_mcpc"
# })
# df_merged = df_2023.merge(as_2023, on="datetime", how="left")
# df_merged

In [ ]:

# dup_as = as_2023[as_2023.duplicated("datetime", keep=False)].sort_values("datetime")
# print("Duplicate AS rows:", len(dup_as))
# dup_as.head(20)

Duplicate AS rows: 2


,datetime,ancillary_type,regdn_mcpc,dst_flag
7393,2023-11-05 02:00:00,REGDN,1.49,N
7394,2023-11-05 02:00:00,REGDN,4.98,Y


In [72]:
df_2024 = load_dam_settlement_prices('data/dam_spp_2024.zip', 2024, 'LZ_HOUSTON')
df_2024

,datetime,SettlementPoint,SettlementPointPrice,DSTFlag
0,2024-01-01 00:00:00,LZ_HOUSTON,15.45,N
1,2024-01-01 01:00:00,LZ_HOUSTON,15.79,N
2,2024-01-01 02:00:00,LZ_HOUSTON,16.81,N
3,2024-01-01 03:00:00,LZ_HOUSTON,16.76,N
4,2024-01-01 04:00:00,LZ_HOUSTON,17.39,N
...,...,...,...,...
8779,2024-12-31 19:00:00,LZ_HOUSTON,35.29,N
8780,2024-12-31 20:00:00,LZ_HOUSTON,32.31,N
8781,2024-12-31 21:00:00,LZ_HOUSTON,30.28,N
8782,2024-12-31 22:00:00,LZ_HOUSTON,24.78,N


2024 was a leap here, hence the 24 additional records

In [ ]:
# Export cleaned DAM settlement prices for use in processing.ipynb

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

csv_2023_path = output_dir / "dam_settlement_prices_LZ_HOUSTON_2023.csv"
csv_2024_path = output_dir / "dam_settlement_prices_LZ_HOUSTON_2024.csv"

df_2023.to_csv(csv_2023_path, index=False)
df_2024.to_csv(csv_2024_path, index=False)

csv_2023_path, csv_2024_path

(PosixPath('data/dam_settlement_prices_LZ_HOUSTON_2023.csv'),
 PosixPath('data/dam_settlement_prices_LZ_HOUSTON_2024.csv'))